# Imports

In [ ]:
import json
import os
import numpy as np
import Levenshtein as lev
import pandas as pd
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## Readme

The prediction file from **tesseract**&**easyocr** should follow the following format:
```json
[
    "Image_name": {
        "ground_truth_1": [
            "prediction_1",
            confidence_score
        ],
        "ground_truth_2": [
            "prediction_2",
            confidence_score
        ]
    },
    ...
]
```
The prediction file from any **LLM** should follow the following format:
```json
[
    "Image_name": [
        "prediction_1",
        "prediction_2",
        ...
        "prediction_n"
    ],
    ...
]
```
The resulting metrics file will follow the following format:
```json
[
    "Image_name": {
        "ground_truth_1": [
            {
                "prediction": "prediction_1",
                "confidence": confidence_score,
                "dist": levenshtein_distance,
                "ratio": levenshtein_ratio,
                "cer": character_error_rate,
                "exact_match": 0 or 1
            },
            ...
        ],
        ...
    },
    ...
]
```

# Parameters

In [ ]:
# Sort by natural order (the alphanumeric order) of the keys in the dictionary
def sort_dict(pathfile):
    d = json.load(open(pathfile))
    # Sort the dictionary by keys in natural order (alphanumeric order) and write it to a new file
    sorted_d = {k: d[k] for k in sorted(d.keys(), key=lambda x: (int(x.split('-')[0]) if x.split('-')[0].isdigit() else float('inf'), x))}
    with open(pathfile, 'w') as f:
        json.dump(sorted_d, f, indent=2)
        f.close()

In [ ]:
# Models: pytess, easyocr, gemma-3-4b-it, LightOnOCR-2-1B, GLM-OCR
MODEL = "pytess"
DATASET_NAME = "ICDAR03/apanar"
DATASET_PATH = f"output/{DATASET_NAME}/"
INPUT_PATH = f"output/predictions/{DATASET_NAME.split('/')[0]}/{DATASET_NAME.split('/')[1]}_{MODEL}.json"
OUTPUT_PATH = f"output/results/{DATASET_NAME}/{MODEL}.json"
is_LLM = False if MODEL in ["pytess", "easyocr"] else True

print(f"Input path: {INPUT_PATH}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Model: {MODEL}")
print(f"Input path exists: {os.path.exists(INPUT_PATH)}")
print(f"Dataset path exists: {os.path.exists(DATASET_PATH)}")

# Metrics

## Definitions

In [ ]:
def clean_float(string):
    # When int are read, they were casted to float ("23": "23.0",)
    try:
        if "." in string and float(string) == int(float(string)):
            return str(int(float(string)))
    except ValueError:
        pass
    return string

def get_words_lists():
    with open(f"{DATASET_PATH}data.json", 'r') as f:
        data = json.load(f)
        
    gt_words = {}
    for img_name, words in data.items():
        words = []
        for word in data[img_name].keys():
            cleaned_word = clean_float(word)
            words.append(cleaned_word)
        gt_words[img_name] = words
    return gt_words

def hungarian_align(gt_tokens, pred_tokens):
    """
    Align GT tokens with predicted tokens using Hungarian matching on
    Levenshtein distance.

    If len(gt_tokens) > len(pred_tokens), missing predictions are modeled
    as empty strings ("") so every GT token gets matched.
    Extra predicted tokens are ignored (no empty GT matching).

    Returns:
        list[tuple[str, str, int]]: (gt_token, pred_token, distance)
    """
    gt_tokens = [str(token).lower() for token in gt_tokens]
    pred_tokens = [str(token).lower() for token in pred_tokens]

    n_gt = len(gt_tokens)
    n_pred = len(pred_tokens)

    if n_gt == 0:
        return []
    if n_pred == 0:
        return [(gt, "", len(gt)) for gt in gt_tokens]

    pad = max(0, n_gt - n_pred)
    pred_aug = pred_tokens + [""] * pad

    cost = np.zeros((n_gt, len(pred_aug)), dtype=np.float32)
    for i, gt in enumerate(gt_tokens):
        for j, pred in enumerate(pred_aug):
            cost[i, j] = lev.distance(gt, pred)

    row_ind, col_ind = linear_sum_assignment(cost)

    mapping = []
    for i, j in zip(row_ind, col_ind):
        pred_word = pred_aug[j]
        mapping.append((gt_tokens[i], pred_word, int(cost[i, j])))

    return mapping

## Evaluation
Based on the predictions and ground truth, we can compute the following metrics:

In [ ]:
predict_data = json.load(open(INPUT_PATH, "r"))
print(f"Loading predictions from {INPUT_PATH}...")
num_predictions = sum(len(words) for words in predict_data.values())
print(f"Loaded predictions for {num_predictions} words in {len(predict_data)} images.")
results = {}
cer_sum = 0
leven_dist_sum = 0
leven_ratio_sum = 0
exact_match_sum = 0

if is_LLM:
    gt_words_dict = get_words_lists()
for image_name, predictions in predict_data.items():
    # if LLM, then predictions is a list of words.
    if is_LLM or isinstance(predictions, list):
        image_name = image_name.replace(".jpg", "").replace(".png", "").replace(".JPG", "").replace(".PNG", "")
        gt_words = gt_words_dict.get(image_name, [])
        
        mapping = hungarian_align(gt_words, predictions)
        
        for gt_word, pred_word, dist in mapping:
            ratio = lev.ratio(gt_word.lower(), pred_word.lower())
            cer = dist / len(gt_word) if len(gt_word) > 0 else 0
            results.setdefault(image_name, {})[gt_word] = {
                "truth": gt_word.lower(),
                "pred": pred_word.lower(),
                "dist": dist,
                "ratio": ratio,
                "cer": cer,
                "exact_match": 1 if gt_word.lower() == pred_word.lower() else 0
            }
            cer_sum += cer
            leven_dist_sum += dist
            leven_ratio_sum += ratio
            exact_match_sum += 1 if gt_word.lower() == pred_word.lower() else 0
    else:
        # if not LLM, then predictions is a dict of words and their probabilities.
        for gt_key, pred in predictions.items():
            if not isinstance(pred[0], list):
                gt_word = gt_key
                word, confidence = pred
                word = clean_float(word)
                word = word.lower().replace(".", "")
                dist = lev.distance(gt_word.lower(), word)
                ratio = round(lev.ratio(gt_word.lower(), word), 4)
                cer = round(dist / len(gt_word) if len(gt_word) > 0 else 0, 4)
            else:
                # If pred[0] is a list, it means there are multiple predictions for the same GT word.
                # We will take the prediction with the highest confidence.
                best_pred = max(pred, key=lambda x: x[1])  # x[1] is the confidence
                gt_word = gt_key
                word, confidence = best_pred
                word = clean_float(word)
                word = word.lower().replace(".", "")
                dist = lev.distance(gt_word.lower(), word)
                ratio = round(lev.ratio(gt_word.lower(), word), 4)
                cer = round(dist / len(gt_word) if len(gt_word) > 0 else 0, 4)
                
            results.setdefault(image_name, {})[gt_key] = {
                "truth": gt_key.lower(),
                "pred": word,
                "confidence": round(confidence, 4),
                "dist": dist,
                "ratio": ratio,
                "cer": cer,
                "exact_match": 1 if gt_word.lower() == word else 0
            }
            cer_sum += cer
            leven_dist_sum += dist
            leven_ratio_sum += ratio
            exact_match_sum += 1 if gt_word.lower() == word else 0
            
# Global metrics
try:
    global_cer = round(cer_sum / num_predictions, 4) if num_predictions > 0 else 0
    global_leven_dist = round(leven_dist_sum / num_predictions, 4) if num_predictions > 0 else 0
    global_leven_ratio = round(leven_ratio_sum / num_predictions, 4) if num_predictions > 0 else 0
    global_exact_match = round(exact_match_sum / num_predictions, 4) if num_predictions > 0 else 0
    
    results["global_metrics"] = {
        "cer": global_cer,
        "leven_dist": global_leven_dist,
        "leven_ratio": global_leven_ratio,
        "exact_match": global_exact_match
    }
    print(f"Global metrics calculated: \nCER={global_cer}, \nLevenshtein Distance={global_leven_dist},\nLevenshtein Ratio={global_leven_ratio}, \nExact Match={global_exact_match}")
except:
    print("Error calculating global metrics. Check if num_predictions is greater than 0.")
    
print(f"Saving results to {OUTPUT_PATH}...")
with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# Metrics Consolidation
We make a dataframe with the following columns:
- `dataset`: The name of the dataset.
- `subset`: The subset of the dataset (e.g., train, test, validation).
- `model`: The name of the model used for predictions.
- `image_name`: The name of the image.
- `ground_truth`: The ground truth text.
- `prediction`: The predicted text.
- `confidence`: The confidence score of the prediction.
- `dist`: The Levenshtein distance between the ground truth and prediction.
- `ratio`: The Levenshtein ratio between the ground truth and prediction.
- `cer`: The character error rate between the ground truth and prediction.
- `exact_match`: A binary indicator of whether the prediction exactly matches the ground truth (1 for match, 0 for no match).

In [ ]:
def build_datagram(results_dir_path: str, dataset_excludes: list[str] = [], subset_excludes: list[str] = []) -> pd.DataFrame:
    rows = []

    for dataset in os.listdir(results_dir_path):
        dataset_path = os.path.join(results_dir_path, dataset)

        if not os.path.isdir(dataset_path):
            continue

        if dataset in dataset_excludes:
            continue

        for subset in os.listdir(dataset_path):
            if subset in subset_excludes:
                continue

            subset_path = os.path.join(dataset_path, subset)

            if not os.path.isdir(subset_path):
                continue

            for file in os.listdir(subset_path):
                file_path = os.path.join(subset_path, file)

                if not file.endswith(".json"):
                    continue

                print(f"Loading results from {file_path}...")
                
                model_name = file.replace(".json", "")

                with open(file_path, "r") as f:
                    data = json.load(f)

                for image_name, words in data.items():
                    if image_name == "global_metrics":
                        continue

                    for gt_word, metrics in words.items():
                        rows.append({
                            "dataset": dataset,
                            "subset": subset,
                            "model": model_name,
                            "image_name": image_name,
                            "gt_word": gt_word,
                            "pred_word": metrics.get("pred", ""),
                            "confidence": metrics.get("confidence", np.nan),
                            "dist": metrics.get("dist", np.nan),
                            "ratio": metrics.get("ratio", np.nan),
                            "cer": metrics.get("cer", np.nan),
                            "exact_match": metrics.get("exact_match", np.nan)
                        })

    return pd.DataFrame(rows)


In [ ]:
exclusion = ["ICDAR19 ArT"] 
metric_df = build_datagram("output/results", dataset_excludes=exclusion)

# Metrics Analysis
Based on the computed metrics, we can analyze the performance of the models and visualize the results.

In [ ]:
def plot_reliability_diagram(confidences, exact_matches, n_bins=10, title="Diagramme de fiabilité"):
    confidences = np.asarray(confidences, dtype=float)
    exact_matches = np.asarray(exact_matches, dtype=float)
    # Ajouter ecrat-type ou nombre d'image par colonne

    if confidences.max() > 1.0:
        confidences = confidences / 100.0

    bin_width = 1.0 / n_bins
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_mids = np.array([bin_edges[i] + bin_width / 2 for i in range(n_bins)])

    bin_indices = np.digitize(confidences, bin_edges, right=True) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    bin_acc_means = np.zeros(n_bins)
    bin_conf_means = np.zeros(n_bins)  # gardé pour le calcul de l'ECE
    bin_counts = np.zeros(n_bins, dtype=int)

    for b in range(n_bins):
        mask = bin_indices == b
        bin_counts[b] = mask.sum()
        if bin_counts[b] > 0:
            bin_acc_means[b] = exact_matches[mask].mean()
            bin_conf_means[b] = confidences[mask].mean()

    n_total = len(confidences)
    ece = np.sum(bin_counts / n_total * np.abs(bin_conf_means - bin_acc_means))

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibration parfaite")

    ax.bar(bin_mids, bin_acc_means, width=bin_width * 0.9, alpha=0.7,
           color="steelblue", edgecolor="black", label="Accuracy observée (EM)", align="center")

    ax.set_xlabel("Bin de confiance")
    ax.set_ylabel("Taux de exact match (EM) observé")
    ax.set_title(f"{title}\nECE = {ece:.4f}")
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks(bin_mids)
    ax.set_xticklabels([f"{m:.3f}" for m in bin_mids], rotation=45)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    return ece

def plot_confidence_vs_cer(confidences, cers, n_bins=10, title="Confiance vs CER"):
    confidences = np.asarray(confidences, dtype=float)
    if confidences.max() > 1.0:
        confidences = confidences / 100.0

    bin_width = 1.0 / n_bins
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_mids = np.array([bin_edges[i] + bin_width / 2 for i in range(n_bins)])

    df = pd.DataFrame({"confidence": confidences, "cer": cers})
    bin_indices = np.digitize(df["confidence"], bin_edges, right=True) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    df["conf_bin_mid"] = bin_mids[bin_indices]

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(data=df, x="conf_bin_mid", y="cer", ax=ax, color="lightsteelblue",
                order=bin_mids)
    ax.set_xlabel("Bin de confiance (centre du bin)")
    ax.set_ylabel("CER (Character Error Rate)")
    ax.set_title(title)
    ax.set_xticklabels([f"{m:.2f}" for m in bin_mids], rotation=45)
    plt.tight_layout()
    plt.show()

    corr = df["confidence"].corr(df["cer"])
    print(f"Corrélation confiance/CER (Pearson): {corr:.4f}")
    return corr

def plot_risk_coverage(confidences, exact_matches, title="Risque vs Couverture"):
    confidences = np.asarray(confidences, dtype=float)
    if confidences.max() > 1.0:
        confidences = confidences / 100.0
    exact_matches = np.asarray(exact_matches, dtype=float)

    order = np.argsort(-confidences)
    sorted_em = exact_matches[order]

    n = len(sorted_em)
    coverage = np.arange(1, n + 1) / n
    cumulative_error_rate = 1 - np.cumsum(sorted_em) / np.arange(1, n + 1)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(coverage, cumulative_error_rate, color="crimson")
    ax.set_xlabel("Couverture (fraction des prédictions gardées, triées par confiance décroissante)")
    ax.set_ylabel("Taux d'erreur cumulé")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
def plot_edit_distance_distribution(distances, title="Distribution des distances d'édition"):
    distances = np.asarray(distances, dtype=int)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.histplot(distances, bins=range(distances.min(), distances.max() + 2), kde=False, color="lightseagreen", ax=ax)
    ax.set_xlabel("Distance d'édition (Levenshtein)")
    ax.set_ylabel("Nombre de mots")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
METRICS_PATH = f"output/results/{DATASET_NAME}/{MODEL}.json"
if not os.path.exists(METRICS_PATH):
    print(f"Metrics file {METRICS_PATH} does not exist. Please run the evaluation first.")

In [ ]:
with open(METRICS_PATH, "r") as f:
    results = json.load(f)

confidences, exact_matches, cers = [], [], []

for image_name, words in results.items():
    if image_name == "averages":
        continue
    for gt_key, metrics in words.items():
        entries = metrics if isinstance(metrics, list) else [metrics]
        for m in entries:
            if not isinstance(m, dict):
                continue
            if "confidence" not in m:
                continue
            confidences.append(float(m["confidence"]))
            exact_matches.append(m["exact_match"])
            cers.append(m["cer"])

ece = plot_reliability_diagram(confidences, exact_matches)
plot_confidence_vs_cer(confidences, cers)
plot_risk_coverage(confidences, exact_matches)
plot_edit_distance_distribution([m["dist"] for image_name, words in results.items() if image_name != "averages" for gt_key, metrics in words.items() for m in (metrics if isinstance(metrics, list) else [metrics]) if isinstance(m, dict) and "dist" in m])